In [1]:
#Load Libraries
library(Seurat)
library(Matrix)
library(future)

#Set Options
options(future.globals.maxSize = 400000 * 1024^2) #for 400GB max size
plan("multicore", workers = 4)

#Set working directory
setwd("/storage1/fs1/jmillman/Active/DigitalTwin")

Loading required package: SeuratObject

Loading required package: sp

‘SeuratObject’ was built with package ‘Matrix’ 1.7.3 but the current
version is 1.7.4; it is recomended that you reinstall ‘SeuratObject’ as
the ABI for ‘Matrix’ may have changed


Attaching package: ‘SeuratObject’


The following objects are masked from ‘package:base’:

    intersect, t




# RNA Integration

## Integrate

In [2]:
obj <- readRDS('checkpoints/DT_RNAmerged.rds')
DefaultAssay(obj) <- "RNA"

In [3]:
obj <- IntegrateLayers(
  object = obj, method = HarmonyIntegration,
  orig.reduction = "pca", new.reduction = "harmony",
  verbose = FALSE)

obj <- FindNeighbors(obj, reduction = "harmony", dims = 1:30, verbose = FALSE)
obj <- FindClusters(obj, resolution = 0.8, cluster.name = "harmony_clusters", verbose = FALSE)
obj <- RunUMAP(obj, reduction = "harmony", dims = 1:30, reduction.name = "umap.harmony", verbose = FALSE)

gc()

The `features` argument is ignored by `HarmonyIntegration`.
This message is displayed once per session.
Warning message:
“Quick-TRANSfer stage steps exceeded maximum (= 16524850)”
Warning message:
“UNRELIABLE VALUE: One of the ‘future.apply’ iterations (‘future_lapply-1’) unexpectedly generated random numbers without declaring so. There is a risk that those random numbers are not statistically sound and the overall results might be invalid. To fix this, specify 'future.seed=TRUE'. This ensures that proper, parallel-safe random numbers are produced via a parallel RNG method. To disable this check, use 'future.seed = NULL', or set option 'future.rng.onMisuse' to "ignore".”
Warning message:
“The default method for RunUMAP has changed from calling Python UMAP via reticulate to the R-native UWOT using the cosine metric
To use Python UMAP via reticulate, set umap.method to 'umap-learn' and metric to 'correlation'
This message will be shown once per session”


,used,(Mb),gc trigger,(Mb),max used,(Mb)
Ncells,4069836,217.4,9276010,495.4,9276010,495.4
Vcells,3984083051,30396.2,6692133585,51057.0,4675050107,35667.9


In [4]:
#obj <- IntegrateLayers(
#  object = obj, method = CCAIntegration,
#  orig.reduction = "pca", new.reduction = "integrated.cca",
#  verbose = FALSE)

#obj <- FindNeighbors(obj, reduction = "integrated.cca", dims = 1:30)
#obj <- FindClusters(obj, resolution = 1.2, cluster.name = "cca_clusters")
#obj <- RunUMAP(obj, reduction = "integrated.cca", dims = 1:30, reduction.name = "umap.integrated.cca")

In [5]:
#obj <- IntegrateLayers(
#  object = obj, method = RPCAIntegration,
#  orig.reduction = "pca", new.reduction = "integrated.rpca",
#  verbose = FALSE)

#obj <- FindNeighbors(obj, reduction = "integrated.rpca", dims = 1:30)
#obj <- FindClusters(obj, resolution = 1.2, cluster.name = "rpca_clusters")
#obj <- RunUMAP(obj, reduction = "integrated.rpca", dims = 1:30, reduction.name = "umap.integrated.rpca")

In [6]:
#obj <- IntegrateLayers(
#  object = obj, method = FastMNNIntegration,
#  new.reduction = "integrated.mnn",
#  verbose = FALSE)

#obj <- FindNeighbors(obj, reduction = "integrated.mnn", dims = 1:30)
#obj <- FindClusters(obj, resolution = 1.2, cluster.name = "mnn_clusters")
#obj <- RunUMAP(obj, reduction = "integrated.mnn", dims = 1:30, reduction.name = "umap.integrated.mnn")

In [7]:
#use_condaenv("shortcake_default", required = TRUE)

#new_obj<-obj

# convert a v5 assay to a v3 assay
#new_obj[["RNA"]] <- as(object = obj[["RNA"]], Class = "Assay")
#andata_obj <- sceasy::convertFormat(new_obj, 
#                       from="seurat",
#                       to="anndata", 
#                       main_layer="counts",
#                       drop_single_values=FALSE)

#sc <- import("scanpy", convert = FALSE)
#scvi <- import("scvi", convert = FALSE)

# run setup_anndata
#scvi$model$SCVI$setup_anndata(andata_obj, batch_key="dataset")
# create the model
#model = scvi$model$SCVI(andata_obj, n_layers=2, n_latent=30, gene_likelihood="nb")
# train the model
#model$train()

# Embedding scvi dims back to Seurat
# get the latent represenation
#latent = model$get_latent_representation()

# put it back in our original Seurat object
#latent <- as.matrix(latent)
#rownames(latent) = colnames(new_obj)
#obj[["integrated.scvi"]] <- CreateDimReducObject(embeddings = latent, key = "scvi_", assay = DefaultAssay(obj))

#rm(list = c('new_obj','andata_obj','sc','scvi','model','latent'))

#obj <- FindNeighbors(obj, reduction = "integrated.scvi", dims = 1:30)
#obj <- FindClusters(obj, resolution = 0.6, cluster.name = "scvi_clusters")
#obj <- RunUMAP(obj, reduction = "integrated.scvi", dims = 1:30, reduction.name = "umap.integrated.scvi")

In [8]:
obj <- JoinLayers(obj)
saveRDS(obj, file="checkpoints/DT_RNAintegrated.rds")

In [9]:
harmony.cluster.markers <- FindAllMarkers(obj, group.by = 'harmony_clusters', only.pos=TRUE, logfc.threshold = 0.5, min.pct = 0.1)
write.csv(harmony.cluster.markers, file = "outputs/ClusterMarkers/DT_RNAintegrated_clustermarkers_res08.csv")

#cca.cluster.markers <- FindAllMarkers(obj, group.by = 'cca_clusters', only.pos=TRUE, logfc.threshold = 0.5, min.pct = 0.1)
#write.csv(cca.cluster.markers, file = "Multiome/SCisletAtlas/GeneLists/RNAcca_clustermarkers_res12.csv")

#rpca.cluster.markers <- FindAllMarkers(obj, group.by = 'rpca_clusters', only.pos=TRUE, logfc.threshold = 0.5, min.pct = 0.1)
#write.csv(rpca.cluster.markers, file = "Multiome/SCisletAtlas/GeneLists/RNArpca_clustermarkers_res12.csv")

#mnn.cluster.markers <- FindAllMarkers(obj, group.by = 'mnn_clusters', only.pos=TRUE, logfc.threshold = 0.5, min.pct = 0.1)
#write.csv(mnn.cluster.markers, file = "Multiome/SCisletAtlas/GeneLists/RNAmnn_clustermarkers_res12.csv")

#scvi.cluster.markers <- FindAllMarkers(obj, group.by = 'scvi_clusters', only.pos=TRUE, logfc.threshold = 0.5)
#write.csv(scvi.cluster.markers, file = "Multiome/SCisletAtlas/GeneLists/RNAscvi_clustermarkers_res06.csv")

Calculating cluster 0

Calculating cluster 1

Calculating cluster 2

Calculating cluster 3

Calculating cluster 4

Calculating cluster 5

Calculating cluster 6

Calculating cluster 7

Calculating cluster 8

Calculating cluster 9

Calculating cluster 10

Calculating cluster 11

Calculating cluster 12

Calculating cluster 13

Calculating cluster 14

Calculating cluster 15

Calculating cluster 16

Calculating cluster 17

Calculating cluster 18

Calculating cluster 19

Calculating cluster 20

Calculating cluster 21

Calculating cluster 22

Calculating cluster 23

Calculating cluster 24

Calculating cluster 25

Calculating cluster 26

Calculating cluster 27

Calculating cluster 28

Calculating cluster 29



In [10]:
sink("Notebooks/3_DigitalTwin/02_DT_RNAintegration-sessionInfo.txt")
sessionInfo()
sink()

R version 4.5.1 (2025-06-13)
Platform: x86_64-pc-linux-gnu
Running under: Ubuntu 22.04.5 LTS

Matrix products: default
BLAS:   /usr/lib/x86_64-linux-gnu/atlas/libblas.so.3.10.3 
LAPACK: /usr/lib/x86_64-linux-gnu/atlas/liblapack.so.3.10.3;  LAPACK version 3.10.0

locale:
 [1] LC_CTYPE=C.UTF-8       LC_NUMERIC=C           LC_TIME=C.UTF-8       
 [4] LC_COLLATE=C.UTF-8     LC_MONETARY=C.UTF-8    LC_MESSAGES=C.UTF-8   
 [7] LC_PAPER=C.UTF-8       LC_NAME=C              LC_ADDRESS=C          
[10] LC_TELEPHONE=C         LC_MEASUREMENT=C.UTF-8 LC_IDENTIFICATION=C   

time zone: Etc/UTC
tzcode source: system (glibc)

attached base packages:
[1] stats     graphics  grDevices utils     datasets  methods   base     

other attached packages:
[1] future_1.67.0      Matrix_1.7-4       Seurat_5.3.1.1000  SeuratObject_5.2.0
[5] sp_2.2-0          

loaded via a namespace (and not attached):
  [1] deldir_2.0-4           pbapply_1.7-4          gridExtra_2.3         
  [4] rlang_1.1.6            magritt